In [ ]:
from __future__ import annotations

import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import torch
from torch import nn


"""
Interactive comparison: PCA plane vs. nonlinear AE manifold in R^3.

- Data: nonlinear 2D manifold embedded in 3D.
- PCA: rank-2 plane and reconstructions.
- Autoencoder: nonlinear encoder and nonlinear decoder, trained with MSE on centered data.
- Visualization: interactive 3D plot with original data, PCA recon, AE recon, PCA plane, and AE manifold surface;
  plus a 2D latent scatter.

Run: python this_file.py
Outputs:
  figs/nonlinear_encoder_nonlinear_decoder/ae_3d.html
  figs/nonlinear_encoder_nonlinear_decoder/latent_2d.html
"""


def make_nonlinear_data(n: int = 2000, noise: float = 0.05, seed: int = 7) -> np.ndarray:
    """
    Generate a nonlinear 2D manifold in R^3 with quadratic and sinusoidal structure, then rotate and offset.

    Parameters
    ----------
    n : int
        Number of samples.
    noise : float
        Observation noise standard deviation.
    seed : int
        Random seed.

    Returns
    -------
    np.ndarray
        Array of shape (n, 3) with dtype float32.
    """
    rng = np.random.default_rng(seed)
    u = rng.uniform(-2.6, 2.6, size=n).astype(np.float32)
    v = rng.uniform(-2.0, 2.0, size=n).astype(np.float32)
    x1 = u
    x2 = v + 0.18 * (u ** 2)
    x3 = 0.55 * (u ** 2) + 0.28 * np.sin(2.3 * v) + 0.22 * u * v
    X = np.stack([x1, x2, x3], axis=1).astype(np.float32)
    R = np.array([[0.36, -0.80, 0.48],
                  [0.80,  0.06, 0.60],
                  [-0.48, 0.60, 0.64]], dtype=np.float32)
    X = X @ R.T
    X += rng.normal(0.0, noise, size=X.shape).astype(np.float32)
    X += np.array([0.8, -0.4, 0.5], dtype=np.float32)
    return X


def pca_2d(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute PCA rank-2 subspace and reconstructions.

    Parameters
    ----------
    X : np.ndarray
        Input of shape (n, 3).

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        mu: (3,), W: (3,2), Z: (n,2), X_hat: (n,3).
    """
    mu = X.mean(axis=0, dtype=np.float32)
    Xc = X - mu
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    V = Vt.T.astype(np.float32)
    W = V[:, :2].astype(np.float32)
    Z = Xc @ W
    X_hat = Z @ W.T + mu
    return mu, W, Z, X_hat


class NonlinearAE(nn.Module):
    """
    Autoencoder with nonlinear encoder and nonlinear decoder operating on centered inputs.
    """

    def __init__(self, d_in: int = 3, d_latent: int = 2, h1: int = 128, h2: int = 128) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, h1),
            nn.GELU(),
            nn.Linear(h1, h2),
            nn.GELU(),
            nn.Linear(h2, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, h2),
            nn.GELU(),
            nn.Linear(h2, h1),
            nn.GELU(),
            nn.Linear(h1, d_in, bias=False),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        Encode centered inputs to latent codes.

        Parameters
        ----------
        x : torch.Tensor
            Tensor of shape (n, 3) representing centered inputs.

        Returns
        -------
        torch.Tensor
            Latent codes of shape (n, 2).
        """
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """
        Decode latent codes to centered reconstructions.

        Parameters
        ----------
        z : torch.Tensor
            Latent codes of shape (n, 2).

        Returns
        -------
        torch.Tensor
            Centered reconstructions of shape (n, 3).
        """
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Encode then decode centered inputs.

        Parameters
        ----------
        x : torch.Tensor
            Centered inputs of shape (n, 3).

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor]
            z: (n, 2), xhat: (n, 3) centered reconstructions.
        """
        z = self.encode(x)
        xhat = self.decode(z)
        return z, xhat


def train_nonlinear_ae(
    X: np.ndarray,
    latent_dim: int = 2,
    epochs: int = 2500,
    lr: float = 1e-3,
    seed: int = 123,
) -> tuple[np.ndarray, NonlinearAE, np.ndarray, np.ndarray]:
    """
    Train the nonlinear encoder/decoder autoencoder with MSE on centered data.

    Parameters
    ----------
    X : np.ndarray
        Input of shape (n, 3).
    latent_dim : int
        Latent dimensionality.
    epochs : int
        Training epochs.
    lr : float
        Learning rate.
    seed : int
        Torch random seed.

    Returns
    -------
    tuple[np.ndarray, NonlinearAE, np.ndarray, np.ndarray]
        mu: (3,), model: trained NonlinearAE, Z: (n,2) latents, X_hat: (n,3) reconstructions.
    """
    torch.manual_seed(seed)
    X_t = torch.tensor(X, dtype=torch.float32)
    mu = X_t.mean(dim=0, keepdim=True)
    Xc = X_t - mu

    model = NonlinearAE(d_in=3, d_latent=latent_dim, h1=128, h2=128)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        opt.zero_grad()
        z, xhat = model(Xc)
        loss = ((xhat - Xc) ** 2).mean()
        loss.backward()
        opt.step()

    with torch.no_grad():
        Z, Xc_hat = model(Xc)
        Z_np = Z.numpy().astype(np.float32)
        X_hat = (Xc_hat + mu).numpy().astype(np.float32)
        mu_np = mu.squeeze(0).numpy().astype(np.float32)

    return mu_np, model, Z_np, X_hat


def make_pca_plane(mu: np.ndarray, W: np.ndarray, Z: np.ndarray, steps: int = 40) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a PCA plane surface from latent grid mapped via x = mu + W z.

    Parameters
    ----------
    mu : np.ndarray
        Mean vector (3,).
    W : np.ndarray
        PCA basis (3,2).
    Z : np.ndarray
        PCA latents (n,2) used for span.
    steps : int
        Grid resolution.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray]
        Xs, Ys, Zs grids for Plotly Surface.
    """
    span = 2.2 * np.std(Z, axis=0)
    u = np.linspace(-span[0], span[0], steps, dtype=np.float32)
    v = np.linspace(-span[1], span[1], steps, dtype=np.float32)
    U, V = np.meshgrid(u, v)
    grid = np.stack([U.ravel(), V.ravel()], axis=1)
    P = grid @ W.T + mu
    Xs = P[:, 0].reshape(steps, steps)
    Ys = P[:, 1].reshape(steps, steps)
    Zs = P[:, 2].reshape(steps, steps)
    return Xs, Ys, Zs


def make_decoder_surface(
    model: NonlinearAE, mu: np.ndarray, Z: np.ndarray, steps: int = 80
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Sample the nonlinear decoder manifold by gridding the latent space and decoding.

    Parameters
    ----------
    model : NonlinearAE
        Trained autoencoder.
    mu : np.ndarray
        Mean vector (3,) to shift centered outputs.
    Z : np.ndarray
        Latents (n,2) used to set the sampling span.
    steps : int
        Grid resolution.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        U, V latent grids and Xs, Ys, Zs surface grids for Plotly Surface.
    """
    span = 2.2 * np.std(Z, axis=0)
    u = np.linspace(-span[0], span[0], steps, dtype=np.float32)
    v = np.linspace(-span[1], span[1], steps, dtype=np.float32)
    U, V = np.meshgrid(u, v)
    grid = np.stack([U.ravel(), V.ravel()], axis=1).astype(np.float32)
    with torch.no_grad():
        z_t = torch.tensor(grid, dtype=torch.float32)
        Xc = model.decode(z_t).numpy().astype(np.float32)
    P = Xc + mu
    Xs = P[:, 0].reshape(steps, steps)
    Ys = P[:, 1].reshape(steps, steps)
    Zs = P[:, 2].reshape(steps, steps)
    return U, V, Xs, Ys, Zs


def plot_interactive(
    X: np.ndarray,
    mu_pca: np.ndarray,
    W_pca: np.ndarray,
    Z_pca: np.ndarray,
    Xhat_pca: np.ndarray,
    mu_ae: np.ndarray,
    Z_ae: np.ndarray,
    Xhat_ae: np.ndarray,
    U: np.ndarray,
    V: np.ndarray,
    Xs_ae: np.ndarray,
    Ys_ae: np.ndarray,
    Zs_ae: np.ndarray,
) -> tuple[go.Figure, go.Figure]:
    """
    Create interactive 3D and 2D Plotly figures showing originals, reconstructions,
    the PCA plane, and the nonlinear AE manifold surface.

    Returns
    -------
    tuple[go.Figure, go.Figure]
        fig3d, fig2d.
    """
    Xs_pca, Ys_pca, Zs_pca = make_pca_plane(mu_pca, W_pca, Z_pca, steps=40)

    fig3d = go.Figure(
        data=[
            go.Scatter3d(x=X[:, 0], y=X[:, 1], z=X[:, 2], mode="markers", name="Original", marker=dict(size=3, opacity=0.35)),
            go.Scatter3d(x=Xhat_pca[:, 0], y=Xhat_pca[:, 1], z=Xhat_pca[:, 2], mode="markers", name="PCA recon", marker=dict(size=3, opacity=0.85)),
            go.Scatter3d(x=Xhat_ae[:, 0], y=Xhat_ae[:, 1], z=Xhat_ae[:, 2], mode="markers", name="AE recon", marker=dict(size=3, opacity=0.85, symbol="diamond")),
            go.Surface(x=Xs_pca, y=Ys_pca, z=Zs_pca, name="PCA plane", showscale=False, opacity=0.25),
            go.Surface(x=Xs_ae, y=Ys_ae, z=Zs_ae, name="AE manifold", showscale=False, opacity=0.35),
        ]
    )
    fig3d.update_layout(
        title="PCA plane vs. Nonlinear AE manifold (R³)",
        scene=dict(xaxis_title="x₁", yaxis_title="x₂", zaxis_title="x₃", aspectmode="data"),
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    fig2d = go.Figure(
        data=[
            go.Scatter(x=Z_pca[:, 0], y=Z_pca[:, 1], mode="markers", name="PCA latent z", marker=dict(size=5, opacity=0.85)),
            go.Scatter(x=Z_ae[:, 0], y=Z_ae[:, 1], mode="markers", name="AE latent z", marker=dict(size=5, opacity=0.7, symbol="diamond")),
        ]
    )
    fig2d.update_layout(
        title="Latent space (R²)",
        xaxis_title="z₁",
        yaxis_title="z₂",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    return fig3d, fig2d


def main() -> None:
    """
    Generate data, fit PCA, train nonlinear AE, sample the decoder manifold, and render interactive plots.
    """
    X = make_nonlinear_data(n=2000, noise=0.05, seed=7)
    mu_pca, W_pca, Z_pca, Xhat_pca = pca_2d(X)
    mu_ae, model, Z_ae, Xhat_ae = train_nonlinear_ae(X, latent_dim=2, epochs=2500, lr=1e-3, seed=123)
    U, V, Xs_ae, Ys_ae, Zs_ae = make_decoder_surface(model, mu_ae, Z_ae, steps=80)
    fig3d, fig2d = plot_interactive(X, mu_pca, W_pca, Z_pca, Xhat_pca, mu_ae, Z_ae, Xhat_ae, U, V, Xs_ae, Ys_ae, Zs_ae)

    outdir = Path("figs").joinpath("nonlinear_encoder_nonlinear_decoder")
    outdir.mkdir(parents=True, exist_ok=True)
    fig3d.write_html(outdir.joinpath("ae_3d.html"), include_plotlyjs="cdn")
    fig2d.write_html(outdir.joinpath("latent_2d.html"), include_plotlyjs="cdn")
    fig3d.show()
    fig2d.show()


if __name__ == "__main__":
    main()